In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
%%capture
!pip install rank-bm25 pyvi pyarrow

In [ ]:
from pathlib import Path
import sys
import json
import re
import time

import numpy as np
import pandas as pd
import torch

TEACHER_MODEL_NAME = "BAAI/bge-reranker-v2-m3"
MAX_LENGTH = 2048
BATCH_SIZE = 32

NEGATIVES_TO_SCORE = 8

BASE_DIR = Path("/content/drive/MyDrive/Legal_Sytem")
DATA_DIR = BASE_DIR / "01_Dataset/02_processed_data"
RETRIEVER_DIR = BASE_DIR / "02_Retriever"

BM25_OUTPUT_DIR = RETRIEVER_DIR / "02_Baseline/BM25/output"
TRAIN_DATA_DIR = RETRIEVER_DIR / "03_Finetuning/train_data"

sys.path.append(str(RETRIEVER_DIR))

import retriever_common as rc

print("Device:", "cuda" if torch.cuda.is_available() else "cpu")

## **01. Read data**

In [ ]:
corpus = pd.read_parquet(DATA_DIR / 'corpus' / 'corpus.parquet')
query = pd.read_parquet(DATA_DIR / 'corpus' / 'queries.parquet')
qrels = pd.read_parquet(DATA_DIR / 'corpus' / 'qrels.parquet')

training_examples = pd.read_parquet(TRAIN_DATA_DIR / 'train_examples.parquet')

bm25_train_ranking = pd.read_parquet(BM25_OUTPUT_DIR / 'train_ranking.parquet')

with open(BM25_OUTPUT_DIR / 'bm25_config.json', encoding='utf-8') as file:
  bm25_config = json.load(file)

doc_id_to_text = dict(zip(corpus['doc_id'], corpus['embedding_text']))
query_id_to_text = dict(zip(query['query_id'], query['query']))

print(f"Mẫu huấn luyện     : {len(training_examples):,}")
print(f"BM25 train_ranking : {len(bm25_train_ranking):,} dòng")
print(f"BM25 config        : k1={bm25_config['k1']}, b={bm25_config['b']}, "
      f"text_column={bm25_config['text_column']}")

### **1.1. List các cặp cần chấm**

In [ ]:
pair_records = []

for row in training_examples.itertuples(index=False):
  pair_records.append({'query_id': row.query_id, 'doc_id': row.positive_doc_id, 'role': 'positive'})

  for doc_id in list(row.negative_doc_ids)[:NEGATIVES_TO_SCORE]:
    pair_records.append({'query_id': row.query_id, 'doc_id': doc_id, 'role': 'negative'})

pairs = pd.DataFrame(pair_records)

# Nếu một doc vừa là positive vừa bị chọn làm negative ở mẫu khác -> ưu tiên nhãn positive.
pairs = (pairs.sort_values('role')
              .drop_duplicates(['query_id', 'doc_id'], keep='first')
              .reset_index(drop=True))

print(f"Cặp (query, doc) duy nhất cần chấm: {len(pairs):,}")
print(pairs['role'].value_counts().to_string())

### **1.2. BM25 score**

In [ ]:
bm25_score_lookup = dict(
    zip(zip(bm25_train_ranking['query_id'], bm25_train_ranking['doc_id']),
        bm25_train_ranking['score'])
)

pairs['bm25_score'] = [
    bm25_score_lookup.get((query_id, doc_id), np.nan)
    for query_id, doc_id in zip(pairs['query_id'], pairs['doc_id'])
]

missing_mask = pairs['bm25_score'].isna()

print(f"Đã có sẵn : {int((~missing_mask).sum()):,}")
print(f"Còn thiếu : {int(missing_mask.sum()):,}  (positive nằm ngoài top-{bm25_config['ranking_depth']})")

In [ ]:
if missing_mask.any():
  from rank_bm25 import BM25Okapi
  from pyvi import ViTokenizer

  def tokenizer_vietnamese(text: str) -> list[str]:
    text = str(text).strip().lower()

    if not text:
      return []

    text = re.sub(r"[^\w\s]", " ", text)

    return ViTokenizer.tokenize(text).split()

  bm25_index = BM25Okapi(
      [tokenizer_vietnamese(text) for text in corpus[bm25_config['text_column']]],
      k1=bm25_config['k1'], b=bm25_config['b'],
  )

  doc_id_to_column = {doc_id: column for column, doc_id in enumerate(corpus['doc_id'])}

  filled_scores = []

  for query_id, doc_id in zip(pairs.loc[missing_mask, 'query_id'], pairs.loc[missing_mask, 'doc_id']):
    scores = bm25_index.get_scores(tokenizer_vietnamese(query_id_to_text[query_id]))
    filled_scores.append(float(scores[doc_id_to_column[doc_id]]))

  pairs.loc[missing_mask, 'bm25_score'] = filled_scores

  print(f"Đã tính bù {len(filled_scores):,} cặp.")

assert pairs['bm25_score'].notna().all(), "Vẫn còn cặp thiếu điểm BM25."

## **3. cross-encoder score**

In [ ]:
from sentence_transformers import CrossEncoder

teacher = CrossEncoder(
    TEACHER_MODEL_NAME,
    max_length=MAX_LENGTH,
    device='cuda' if torch.cuda.is_available() else 'cpu',
    model_kwargs={'torch_dtype': 'float16'} if torch.cuda.is_available() else None,
    trust_remote_code=True,
)

# Kiểm tra không passage nào bị cắt — teacher chấm trên bản cắt cụt là nhãn sai.
longest_token_count = len(teacher.tokenizer.encode(
    max(doc_id_to_text.values(), key=len), add_special_tokens=True
))

print(f"Passage dài nhất: {longest_token_count:,} token / max_length = {teacher.max_length:,}")

assert longest_token_count <= teacher.max_length, "Tăng MAX_LENGTH."

In [ ]:
pair_list = [
    (query_id_to_text[query_id], doc_id_to_text[doc_id])
    for query_id, doc_id in zip(pairs['query_id'], pairs['doc_id'])
]

start_time = time.perf_counter()

pairs['cross_encoder_score'] = np.asarray(
    teacher.predict(pair_list, batch_size=BATCH_SIZE, show_progress_bar=True), dtype=np.float32
)

scoring_seconds = time.perf_counter() - start_time

print(f"\nĐã chấm {len(pairs):,} cặp trong {scoring_seconds / 60:.1f} phút "
      f"({len(pairs) / scoring_seconds:.0f} cặp/giây)")

## **4. Ensemble score**

In [ ]:
def z_score_within_query(dataframe: pd.DataFrame, column: str) -> pd.Series:
  grouped = dataframe.groupby('query_id')[column]

  mean = grouped.transform('mean')
  std = grouped.transform('std').replace(0, np.nan)

  return ((dataframe[column] - mean) / std).fillna(0.0)


pairs['bm25_z'] = z_score_within_query(pairs, 'bm25_score')
pairs['cross_encoder_z'] = z_score_within_query(pairs, 'cross_encoder_score')

pairs['ensemble_score'] = (pairs['bm25_z'] + pairs['cross_encoder_z']) / 2.0

display(pairs[['bm25_score', 'cross_encoder_score', 'bm25_z', 'cross_encoder_z', 'ensemble_score']]
        .describe(percentiles=[0.25, 0.5, 0.75]).round(3))

### **4.1. Check label quality**

In [ ]:
score_by_role = pairs.groupby('role')[['bm25_score', 'cross_encoder_score', 'ensemble_score']].mean()

display(score_by_role.round(4))

for column in ['bm25_score', 'cross_encoder_score', 'ensemble_score']:
  assert score_by_role.loc['positive', column] > score_by_role.loc['negative', column], (
      f"{column}: positive không cao hơn negative — nhãn teacher có vấn đề."
  )

top1_records = []

for column in ['bm25_score', 'cross_encoder_score', 'ensemble_score']:
  best_role = (pairs.sort_values(column, ascending=False)
                    .groupby('query_id')['role'].first())

  top1_records.append({'teacher': column, 'positive_ranked_first': round((best_role == 'positive').mean(), 4)})

display(pd.DataFrame(top1_records))

In [ ]:
correlation = pairs[['bm25_z', 'cross_encoder_z']].corr(method='spearman').iloc[0, 1]

print(f"Spearman(BM25, cross-encoder) = {correlation:.4f}")

if correlation > 0.9:
  print("Hai teacher gần như trùng nhau -> ensemble ít giá trị, và cũng cho thấy cross-encoder chủ yếu bắt trùng từ khoá giống BM25.")
else:
  print("Hai teacher khác nhau đáng kể -> ensemble đáng thử, và cross-encoder sẽ mang tín hiệu ngữ nghĩa mà BM25 không có.")

### **4.2. margin distribution**

In [ ]:
best_positive_score = (pairs[pairs['role'] == 'positive']
                       .groupby('query_id')[['bm25_score', 'cross_encoder_score', 'ensemble_score']]
                       .max())

margin_records = []

for column in ['bm25_score', 'cross_encoder_score', 'ensemble_score']:
  negatives = pairs[pairs['role'] == 'negative'].copy()

  negatives['margin'] = (
      negatives['query_id'].map(best_positive_score[column]) - negatives[column]
  )

  negatives = negatives.dropna(subset=['margin'])

  margin_records.append({
      'teacher': column,
      'mean': round(negatives['margin'].mean(), 4),
      'p05': round(negatives['margin'].quantile(0.05), 4),
      'p50': round(negatives['margin'].quantile(0.50), 4),
      'p95': round(negatives['margin'].quantile(0.95), 4),
      'max_abs': round(negatives['margin'].abs().max(), 4),
      'negative_margin_ratio': round((negatives['margin'] < 0).mean(), 4),
  })

display(pd.DataFrame(margin_records))

## **05. Export data**

In [ ]:
teacher_scores = pairs[[
    'query_id', 'doc_id', 'role',
    'bm25_score', 'cross_encoder_score',
    'bm25_z', 'cross_encoder_z', 'ensemble_score',
]].copy()

teacher_scores.to_parquet(TRAIN_DATA_DIR / 'teacher_scores.parquet', index=False)

scoring_config = {
    'teacher_model': TEACHER_MODEL_NAME,
    'max_length': MAX_LENGTH,
    'negatives_scored': NEGATIVES_TO_SCORE,
    'number_of_pairs': len(teacher_scores),
    'scoring_minutes': round(scoring_seconds / 60, 2),
    'bm25_config': {key: bm25_config[key] for key in ['k1', 'b', 'text_column']},
    'spearman_bm25_vs_cross_encoder': float(correlation),
    'score_columns': {
        'L1': 'bm25_score',
        'L2': 'cross_encoder_score',
        'L1+L2': 'ensemble_score',
    },
}

with open(TRAIN_DATA_DIR / 'teacher_scoring_config.json', 'w', encoding='utf-8') as file:
  json.dump(scoring_config, file, ensure_ascii=False, indent=2)

print(json.dumps(scoring_config, ensure_ascii=False, indent=2))